# Customer Lifetime Value (CLV) Prediction: Machine Learning Approach

This notebook covers **Phase 18: Customer Lifetime Value (CLV) Modeling**.

### Why Machine Learning for CLV?
While classical probabilistic models (like BG/NBD and Gamma-Gamma) are useful, they only use transactional history (Recency, Frequency, Monetary). Modern machine learning models allow us to integrate:
1. **Demographic features** (Country, Gender, Marital Status).
2. **Detailed behavioral features** (Average ticket size, order spacing).
3. **Non-linear relationships** and interactions.

### Methodology:
- **Calibration Period**: Dec 2010 to May 31, 2013 (features calculated here).
- **Holdout Period**: June 1, 2013 to Jan 2014 (target calculated here).
- **Model**: Train a `GradientBoostingRegressor` and `RandomForestRegressor` to predict holdout spending based on calibration features.

## 1. Setup & Load Data

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

db_path = os.path.join("..", "data", "processed", "retail_analytics.db")
conn = duckdb.connect(db_path)

## 2. Feature Engineering (Calibration vs. Holdout)

We will write a SQL query to partition sales into calibration (features) and holdout (target).

In [ ]:
query = """
WITH calibration_features AS (
    SELECT 
        f.customer_key,
        DATE_DIFF('day', MIN(f.order_date), '2013-05-31') AS T,
        DATE_DIFF('day', MIN(f.order_date), MAX(f.order_date)) AS recency,
        COUNT(DISTINCT f.order_number) AS frequency,
        SUM(f.sales_amount) AS monetary,
        SUM(f.sales_amount) / COUNT(DISTINCT f.order_number) AS avg_order_value
    FROM gold.fact_sales f
    WHERE f.order_date <= '2013-05-31'
    GROUP BY f.customer_key
),
holdout_target AS (
    SELECT 
        f.customer_key,
        SUM(f.sales_amount) AS holdout_spending
    FROM gold.fact_sales f
    WHERE f.order_date > '2013-05-31'
    GROUP BY f.customer_key
)
SELECT 
    c.customer_key,
    cust.country,
    cust.gender,
    cust.marital_status,
    c.T,
    c.recency,
    c.frequency,
    c.monetary,
    c.avg_order_value,
    COALESCE(h.holdout_spending, 0) AS target_holdout_spending
FROM calibration_features c
JOIN gold.dim_customers cust ON c.customer_key = cust.customer_key
LEFT JOIN holdout_target h ON c.customer_key = h.customer_key;
"""

df_clv = conn.execute(query).fetchdf()
print(f"Loaded {df_clv.shape[0]} customers active during calibration.")
df_clv.head()

## 3. Data Preprocessing

One-hot encode categorical features (country, gender, marital status).

In [ ]:
# Select features and target
cat_cols = ['country', 'gender', 'marital_status']
num_cols = ['T', 'recency', 'frequency', 'monetary', 'avg_order_value']

# Create dummies
df_processed = pd.get_dummies(df_clv[cat_cols + num_cols], columns=cat_cols, drop_first=True)
X = df_processed.drop(columns=[]) # Keep all columns as features
y = df_clv['target_holdout_spending']

# Split into Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## 4. Model Training & Evaluation

In [ ]:
# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=8)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# Train Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=5)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

# Evaluate Random Forest
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

# Evaluate Gradient Boosting
rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb = r2_score(y_test, y_pred_gb)

print(f"Random Forest:      RMSE = {rmse_rf:.2f}, R2 = {r2_rf:.4f}")
print(f"Gradient Boosting:  RMSE = {rmse_gb:.2f}, R2 = {r2_gb:.4f}")

### Feature Importance Breakdown

In [ ]:
feat_importances = pd.Series(gb.feature_importances_, index=X.columns)
feat_importances.nlargest(10).plot(kind='barh', color='teal')
plt.title("Top 10 Feature Importances for CLV Prediction")
plt.xlabel("Relative Importance")
plt.show()

## 5. Predicting CLV for all customers

We will now use the Gradient Boosting model to predict the expected future value for all active customers, and save the predicted CLV to our segments table.

In [ ]:
# Predict on the entire dataset
df_clv['predicted_future_value'] = gb.predict(X)

# Total CLV = Current Monetary + Predicted Future Value
df_clv['customer_lifetime_value'] = df_clv['monetary'] + df_clv['predicted_future_value']

df_clv_save = df_clv[['customer_key', 'predicted_future_value', 'customer_lifetime_value']].copy()
df_clv_save.head()

In [ ]:
# Save predicted CLV values back to DuckDB
conn.execute("CREATE TEMPORARY TABLE temp_clv AS SELECT * FROM df_clv_save")

# Update the customer segments table in gold schema to include CLV predictions
conn.execute("""
    ALTER TABLE gold.customer_segments ADD COLUMN IF NOT EXISTS predicted_future_value DOUBLE;
    ALTER TABLE gold.customer_segments ADD COLUMN IF NOT EXISTS customer_lifetime_value DOUBLE;
""")
conn.execute("""
    UPDATE gold.customer_segments 
    SET 
        predicted_future_value = (SELECT predicted_future_value FROM temp_clv WHERE temp_clv.customer_key = gold.customer_segments.customer_key),
        customer_lifetime_value = (SELECT customer_lifetime_value FROM temp_clv WHERE temp_clv.customer_key = gold.customer_segments.customer_key);
""")
print("Updated gold.customer_segments table with CLV predictions!")

# Show sample of updated segments
print(conn.execute("SELECT segment_name, COUNT(*) AS customer_count, ROUND(AVG(customer_lifetime_value), 1) AS avg_clv FROM gold.customer_segments GROUP BY segment_name").fetchdf())

conn.close()